<h1 style="text-align:center">Emergencia</h1>
<h3 style="text-align:center">Procesamiento de las búsquedas por motor (final)</h3>
<p style="text-align:center">Se muestran las noticias en un rango definido de tiempo</p>

In [1]:
from unidecode import unidecode
import pandas as pd
import re

pd.set_option('display.max_rows', None, 'display.max_columns', None, 'display.max_colwidth', None)

<h3>Apertura y acondicionamiento del libro</h3>

In [2]:
dr = pd.read_excel('noticias_vzla_lluvias.xlsx')

agregar = ['title_clean', 'snippet_full_clean', 'red_social']
dr[agregar] = None, None, None

print(dr.shape, dr.columns.tolist(), '\n')
dr.tail(1)

(79, 22) ['source_url', 'post_url', 'profile_name', 'username', 'text', 'date', 'likes', 'comments', '_collected_at', 'page_number', 'rank', 'title', 'url', 'displayed_url', 'fecha', 'snippet', 'snippet_full', 'rich_type', 'criterio_busqueda', 'title_clean', 'snippet_full_clean', 'red_social'] 



,source_url,post_url,profile_name,username,text,date,likes,comments,_collected_at,page_number,rank,title,url,displayed_url,fecha,snippet,snippet_full,rich_type,criterio_busqueda,title_clean,snippet_full_clean,red_social
78,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2,25,Largas colas y enfrentamientos: se agudiza la escasez de gasolina en ...,https://eldiario.com/2024/12/06/largas-colas-enfrentamientos-se-agudiza-escasez-gasolina-barinas/,NaN,NaN,"6 dic 2024En junio de 2024, Nicolás Maduro ordenó al entonces ministro de Petróleo, Pedro Tellechea acabar con la escasez de gasolina en el estado Barinas. La exigencia la hizo en el contexto de su campaña electoral.","6 dic 2024En junio de 2024, Nicolás Maduro ordenó al entonces ministro de Petróleo, Pedro Tellechea acabar con la escasez de gasolina en el estado Barinas. La exigencia la hizo en el contexto de su campaña electoral.",NaN,"venezuela ""Diario La Nación"" ""barinas"" (lluvia OR crecida OR inundacion OR deslizamiento OR desbord OR draga OR precipitacion OR vaguada OR tormenta OR damnificados OR afectados) -filetype:pdf -filetype:doc -filetype:xls -scribd after:2026-07-03 before:2026-07-04",None,None,None


<h3>Diccionarios</h3>

In [3]:
emergencias = {
    # --------------------------------------------------
    # Zonas geográficas y demográficas
    # --------------------------------------------------
    'capital': ['caracas', 'miranda', 'distrito capital', 'vargas', 'la guaira', 'chacao', 'baruta', 'el hatillo', 'petare', 'catia', 'gran caracas'],
    'andes': ['merida', 'tachira', 'trujillo', 'bocono', 'valera', 'san cristobal', 'el vigia', 'paramo', 'andes', 'cordillera andina'],
    'oriente': ['sucre', 'anzoategui', 'monagas', 'nueva esparta', 'margarita', 'cumana', 'lecheria', 'maturin', 'carupano', 'oriente'],
    'occidente': ['zulia', 'falcon', 'lara', 'yaracuy', 'maracaibo', 'punto fijo', 'barquisimeto', 'coro', 'occidente'],
    'llanos': ['apure', 'barinas', 'portuguesa', 'cojedes', 'guarico', 'san fernando', 'guanare', 'calabozo', 'llanos', 'apureño'],
    'costa': ['costa', 'litoral', 'playa', 'mar', 'choroni', 'ocumare', 'puerto cabello', 'costero', 'morrocoy'],
    'barrio': ['barrio', 'cerro', 'sector popular', 'rancho', 'vivienda precaria', 'asentamiento', 'comunidad', 'invasión'],
    'urbanizacion': ['urbanizacion', 'residencias', 'edificio', 'condominio', 'quinta', 'calle', 'avenida', 'conjunto'],

    # --------------------------------------------------
    # Nivel de afectación / Consecuencias
    # --------------------------------------------------
    'daños': ['daño', 'destruccion', 'colapso', 'escombros', 'perdida', 'ruina', 'afectacion', 'grieta', 'desplome', 'siniestro', 'viviendas afectadas'],
    'victimas': ['victima', 'fallecido', 'muerto', 'herido', 'lesionado', 'desaparecido', 'damnificado', 'cadaver', 'atrapado', 'deceso'],
    'rescate': ['rescate', 'salvamento', 'evacuacion', 'bomberos', 'proteccion civil', 'paramedicos', 'busqueda', 'sobreviviente', 'inparques', 'voluntarios'],

    # --------------------------------------------------
    # Medios de transporte, logística y maquinaria
    # --------------------------------------------------
    'aereo': ['helicoptero', 'avion', 'dron', 'uav', 'super puma', 'cougar', 'hercules', 'avioneta'],
    'acuatico': ['lancha', 'bote', 'zodiac', 'peñero', 'curiara', 'canoa', 'buque', 'patrullera', 'salvavidas', 'acuatico'],
    'terrestre': ['ambulancia', 'rustico', 'jeep', 'machito', 'camion', 'convoy', 'retroexcavadora', 'tractor', 'payloader', 'grua', 'autobus', '4x4', 'vehiculo'],

    # --------------------------------------------------
    # Atención, salud y refugio
    # --------------------------------------------------
    'atencion': ['refugio', 'albergue', 'cancha', 'iglesia', 'hospital', 'clinica', 'cdi', 'ambulatorio', 'carpa', 'campamento', 'triaje', 'acopio', 'donacion', 'insumos', 'medicinas', 'alimentos', 'agua potable'],

    # --------------------------------------------------
    # Tipos de eventos y desastres
    # --------------------------------------------------
    'eventos': ['sismo', 'terremoto', 'temblor', 'replica', 'escala richter', 'inundacion', 'anegacion', 'desbordamiento', 'crecida', 'deslave', 'derrumbe', 'deslizamiento', 'alud', 'barro', 'aguacero', 'tormenta', 'precipitaciones', 'vaguada', 'lluvia', 'onda tropical']
}

americanos = {'bra': ['brasil','brazil'],
              'chi': ['chile'],
              'arg': ['argentin','buenos aires'],
              'col': ['colombia','medellin','bogota'],
              'pan': ['panama','panameñ'],
              'per': ['peru'],
              'uru': ['uruguay','montevideo'],
              'ecu': ['ecuador','ecuatorian', 'manabi'],
              'prc': ['puerto rico','puertoriqueñ'],
              'crc': ['costa rica','costaricense'],
              'usa': ['estados unidos','eeuu','estadounidense'],
              'esp': ['españa','español'],
              'cub': ['cuba','cuban'],
              'mex': ['mexico','mexican'],
              'rdo': ['republica dominicana','dominican'],
              'gua': ['guatemala','guatemaltec'],
              'bar': ['barbados','barbadens'],
              'bol': ['bolivia'],
              'par': ['paraguay'],
              'sal': ['el salvador','salvadoreñ'],
              'tri': ['trinidad y tobago','trinitari'],
              'bel': ['belice'],
              'hon': ['honduras','hondureñ'],
              'nic': ['nicaragua','nicaragüense','nicaraguense'],
              'gra': ['granada','granadin'],
              'guy': ['guyana','guyanes'],
              'sur': ['surinam'],
              'hai': ['haiti'],
              'can': ['canad'],
              'cur': ['curaz'],
              'aru': ['aruba', 'arube']}

otros = r'espana|mar caribe|europa|italia|nueva segovia|cordoba|francia| nong|veracruz|valparaiso|yucatan|portugal|venecia|monte alegre|tenerife|vietnam|australia|america del sur|rio de janeiro|miami|istmo caribe| lima|madrid|baja california|bariloche|santo domingo|manongo|alaska|america|santa fe|el quisco|aruba|antioquia|costa azul|santiago del estero|san jose|punta palma| china|santiago de chile|valle del cauca|cartagena|canada|sudamerica|alemania|cucuta|rusia|grecia|curazao|nahuel huapi|destructor|armada|militar|narcotrafico|ataque|trump|next day cargo|matanzas| fallec|terrorista|criminal|atentado|explosion|incendio|cuartel moncada|petrole| migra| migro| inmigr| emigr'
valores = []
for k in americanos.values():
    valores.extend(k) 

extra = 'shop| tienda|trabajo|empleo|vacante|contratamos|postula| curso| clase| beca|inscripci[oó]n|chavismo|oposici[oó]n|elecciones|protest|manifest|notici|asesin| robo|secuestr| herido|balacera|accident|emergencia|sexo|porn| eroti|sensual|concierto|[aá]lbum|periodico|unlimited|4x2|4x4|multiventas|repuestos| envio|todo jeep|envios maritimos|expobici|endoparasitos|resonancia'

amer = otros + '|' + '|'.join(valores) + '|' + extra

estados = ['amazonas', 'anzoategui', 'apure', 'aragua', 'barinas', 'bolivar', 'carabobo', 'cojedes', 'delta amacuro', 'dependencias federales', 
           'distrito capital', 'falcon', 'guarico', 'lara', 'la guaira', 'merida', 'miranda', 'monagas', 'nueva esparta', 'portuguesa', 'sucre', 
           'tachira', 'trujillo', 'yaracuy', 'zulia']

municipios = ['alto orinoco', 'atures', 'autana', 'manapiare', 'maroa', 'rio negro', 'anaco', 'aragua', 'manuel ezequiel bruzual', 'diego bautista urbaneja', 'fernando penalver', 'francisco del carmen carvajal', 'general sir arthur mcgregor', 'guanta', 'independencia', 'jose gregorio monagas', 'juan antonio sotillo', 'juan manuel cajigal', 'francisco de miranda', 'pedro maria freites', 'piritu', 'san jose de guanipa', 'san juan de capistrano', 'santa ana', 'simon bolivar', 'simon rodriguez', 'achaguas', 'biruaca', 'munoz', 'paez', 'pedro camejo', 'romulo gallegos', 'san fernando', 'atanasio girardot', 'bolivar', 'camatagua', 'francisco linares alcantara', 'jose angel lamas', 'jose felix ribas', 'jose rafael revenga', 'libertador', 'mario briceno iragorry', 'ocumare de la costa de oro', 'san casimiro', 'san sebastian', 'santiago marino', 'santos michelena', 'urdaneta', 'zamora', 'alberto arvelo torrealba', 'andres eloy blanco', 'antonio jose de sucre', 'arismendi', 'barinas', 'cruz paredes', 'ezequiel zamora', 'obispos', 'pedraza', 'rojas', 'sosa', 'caroni', 'cedeno', 'el callao', 'gran sabana', 'heres', 'piar', 'angostura', 'roscio', 'sifontes', 'padre pedro chien', 'bejuma', 'carlos arvelo', 'diego ibarra', 'guacara', 'juan jose mora', 'los guayos', 'miranda', 'montalban', 'naguanagua', 'puerto cabello', 'san diego', 'san joaquin', 'valencia', 'anzoategui', 'tinaquillo', 'girardot', 'lima blanco', 'pao de san juan bautista', 'ricaurte', 'san carlos', 'tinaco', 'antonio diaz', 'casacoima', 'pedernales', 'tucupita', 'buchivacoa', 'cacique manaure', 'carirubana', 'colina', 'dabajuro', 'falcon', 'federacion', 'jacura', 'jose laurencio silva', 'los taques', 'mauroa', 'monsenor iturriza', 'palmasola', 'petit', 'san francisco', 'tocopero', 'union', 'urumaco', 'camaguan', 'chaguaramas', 'el socorro', 'jose tadeo monagas', 'juan german roscio', 'julian mellado', 'las mercedes', 'leonardo infante', 'pedro zaraza', 'ortiz', 'san geronimo de guayabal', 'san jose de guaribe', 'santa maria de ipire', 'sebastian francisco de miranda', 'crespo', 'iribarren', 'jimenez', 'moran', 'palavecino', 'simon planas', 'torres', 'alberto adriani', 'andres bello', 'antonio pinto salinas', 'aricagua', 'arzobispo chacon', 'campo elias', 'caracciolo parra olmedo', 'cardenal quintero', 'guaraque', 'julio cesar salas', 'justo briceno', 'obispo ramos de lora', 'padre noguera', 'pueblo llano', 'rangel', 'rivas davila', 'santos marquina', 'tulio febres cordero', 'zea', 'acevedo', 'baruta', 'brion', 'buroz', 'carrizal', 'chacao', 'cristobal rojas', 'el hatillo', 'guaicaipuro', 'lander', 'los salias', 'paz castillo', 'pedro gual', 'plaza', 'aguasay', 'caripe', 'maturin', 'punceres', 'santa barbara', 'sotillo', 'uracoa', 'antolin del campo', 'garcia', 'gomez', 'maneiro', 'marcano', 'marino', 'peninsula de macanao', 'tubores', 'villalba', 'diaz', 'agua blanca', 'araure', 'esteller', 'guanare', 'guanarito', 'monsenor jose vicente de unda', 'ospino', 'papelon', 'san genaro de boconoito', 'san rafael de onoto', 'santa rosalia', 'turen', 'andres mata', 'benitez', 'bermudez', 'cajigal', 'cruz salmeron acosta', 'mejia', 'montes', 'ribero', 'valdez', 'antonio romulo costa', 'ayacucho', 'cardenas', 'cordoba', 'fernandez feo', 'garcia de hevia', 'guasimos', 'jauregui', 'jose maria vargas', 'junin', 'lobatera', 'michelena', 'panamericano', 'pedro maria urena', 'rafael urdaneta', 'samuel dario maldonado', 'san cristobal', 'seboruco', 'torbes', 'uribante', 'san judas tadeo', 'bocono', 'candelaria', 'carache', 'escuque', 'jose felipe marquez canizalez', 'juan vicente campos elias', 'la ceiba', 'monte carmelo', 'motatan', 'pampan', 'pampanito', 'rafael rangel', 'san rafael de carvajal', 'trujillo', 'valera', 'vargas', 'aristides bastidas', 'bruzual', 'cocorote', 'jose antonio paez', 'la trinidad', 'manuel monge', 'nirgua', 'pena', 'san felipe', 'urachiche', 'jose joaquin veroes', 'almirante padilla', 'baralt', 'cabimas', 'catatumbo', 'colon', 'francisco javier pulgar', 'jesus enrique losada', 'jesus maria semprun', 'la canada de urdaneta', 'lagunillas', 'machiques de perija', 'mara', 'maracaibo', 'rosario de perija', 'santa rita', 'valmore rodriguez']

parroquias = ['huachamacare acanana', 'marawaka toky shamanana', 'mavaka mavaka', 'sierra parima parimabe', 'ucata laja lisa', 'yapacana macuruco', 'caname guarinuma', 'fernando giron tovar', 'luis alberto gomez', 'pahuena limon de parhuena', 'platanillal platanillal', 'samariapo', 'sipapo', 'munduapo', 'guayapo', 'alto ventuari', 'medio ventuari', 'bajo ventuari', 'victorino', 'comunidad', 'casiquiare', 'cocuy', 'san carlos de rio negro', 'solano', 'cachipo', 'aragua de barcelona', 'lecheria', 'el morro', 'puerto piritu', 'san miguel', 'valle de guanape', 'el chaparro', 'tomas alfaro', 'calatrava', 'chorreron', 'mamo', 'soledad', 'mapire', 'santa clara', 'san diego de cabrutica', 'uverito', 'zuata', 'puerto la cruz', 'pozuelos', 'onoto', 'san pablo', 'san mateo', 'el carito', 'santa ines', 'la romerena', 'atapirire', 'boca del pao', 'el pao', 'pariaguan', 'cantaura', 'santa rosa', 'urica', 'boca de uchire', 'boca de chavez', 'pueblo nuevo', 'bergantin', 'caigua', 'el carmen', 'el pilar', 'naricual', 'san crsitobal', 'edmundo barrios', 'miguel otero silva', 'apurito', 'el yagual', 'guachara', 'mucuritas', 'queseras del medio', 'mantecal', 'quintero', 'rincon hondo', 'san vicente', 'guasdualito', 'aramendi', 'el amparo', 'san camilo', 'san juan de payara', 'codazzi', 'cunaviche', 'elorza', 'el recreo', 'penalver', 'san rafael de atamaica', 'pedro jose ovalles', 'joaquin crespo', 'jose casanova godoy', 'madre maria de san jose', 'los tacarigua', 'las delicias', 'choroni', 'carmen de cura', 'mosenor feliciano gonzalez', 'santa cruz', 'castor nieves rios', 'las guacamayas', 'pao de zarate', 'palo negro', 'san martin de porres', 'el limon', 'cana de azucar', 'ocumare de la costa', 'guiripa', 'ollas de caramacate', 'valle morin', 'san sebastian', 'turmero', 'arevalo aponte', 'chuao', 'saman de guere', 'alfredo pacheco miranda', 'tiara', 'cagua', 'bella vista', 'las penitas', 'san francisco de cara', 'taguay', 'magdaleno', 'san francisco de asis', 'valles de tucutunemo', 'augusto mijares', 'sabaneta', 'juan antonio rodriguez dominguez', 'el canton', 'santa cruz de guacas', 'puerto vivas', 'ticoporo', 'nicolas pulido', 'guadarrama', 'la union', 'san antonio', 'alberto arvelo larriva', 'san silvestre', 'santa lucia', 'torumos', 'romulo betancourt', 'corazon de jesus', 'ramon ignacio mendez', 'alto barinas', 'manuel palacio fajardo', 'dominga ortiz de paez', 'barinitas', 'altamira de caceres', 'calderas', 'barrancas', 'mazparrito', 'pedro briceno mendez', 'jose ignacio del pumar', 'guasimitos', 'el real', 'la luz', 'ciudad bolivia', 'jose ignacio briceno', 'dolores', 'palacio fajardo', 'ciudad de nutrias', 'el regalo', 'puerto nutrias', 'santa catalina', 'cachamay', 'chirica', 'dalla costa', 'once de abril', 'unare', 'universidad', 'vista al sol', 'pozo verde', 'yocoima', '5 de julio', 'altagracia', 'ascension farreras', 'guaniamo', 'la urbana', 'pijiguaos', 'ikabaru', 'catedral', 'orinoco', 'marhuanta', 'agua salada', 'vista hermosa', 'la sabanita', 'panapana', 'pedro cova', 'raul leoni', 'barceloneta', 'salom', 'san isidro', 'aripao', 'guarataro', 'las majadas', 'moitaco', 'rio grande', 'canoabo', 'guigue', 'carabobo', 'tacarigua', 'mariara', 'aguas calientes', 'ciudad alianza', 'yagua', 'moron', 'tocuyito', 'bartolome salom', 'fraternidad', 'goaigoaza', 'juan jose flores', 'borburata', 'patanemo', 'miguel pena', 'san blas', 'san jose', 'negro primero', 'cojedes', 'juan de mata suarez', 'el baul', 'la aguadita', 'macapo', 'libertad de cojedes', 'san carlos de austria', 'juan angel bravo', 'manuel manrique', 'general en jefe jose laurencio silva', 'curiapo', 'almirante luis brion', 'francisco aniceto lugo', 'manuel renaud', 'padre barral', 'santos de abelgas', 'imataca', 'cinco de julio', 'juan bautista arismendi', 'manuel piar', 'luis beltran prieto figueroa', 'jose vidal marcano', 'juan millan', 'leonardo ruiz pineda', 'mariscal antonio jose de sucre', 'monsenor argimiro garcia', 'san rafael', 'virgen del valle', 'clarines', 'guanape', 'sabana de uchire', 'capadare', 'la pastora', 'san juan de los cayos', 'aracua', 'la pena', 'san luis', 'bariro', 'borojo', 'capatarida', 'guajiro', 'seque', 'zazarida', 'valle de eroa', 'urbana punta cardon', 'la vela de coro', 'acurigua', 'guaibacoa', 'las calderas', 'macoruca', 'agua clara', 'avaria', 'pedregal', 'piedra grande', 'purureche', 'adaure', 'adicora', 'baraived', 'buena vista', 'jadacaquiva', 'el vinculo', 'el hato', 'moruy', 'agua larga', 'el pauji', 'maparari', 'agua linda', 'araurima', 'tucacas', 'boca de aroa', 'judibana', 'mene de mauroa', 'san felix', 'casigua', 'guzman guillermo', 'mitare', 'rio seco', 'san gabriel', 'boca del tocuyo', 'chichiriviche', 'tocuyo de la costa', 'cabure', 'curimagua', 'san jose de la costa', 'pecaya', 'el charal', 'las vegas del tuy', 'santa cruz de bucaral', 'puerto cumarebo', 'la cienaga', 'la soledad', 'pueblo cumarebo', 'churuguara', 'puerto miranda', 'tucupido', 'san rafael de laya', 'altagracia de orituco', 'san rafael de orituco', 'san francisco javier de lezama', 'paso real de macaira', 'carlos soublette', 'san francisco de macaira', 'libertad de orituco', 'cantaclaro', 'san juan de los morros', 'parapara', 'el sombrero', 'cabruta', 'santa rita de manapire', 'valle de la pascua', 'espino', 'san jose de unare', 'zaraza', 'san jose de tiznados', 'san francisco de tiznados', 'san lorenzo de tiznados', 'ortiz', 'guayabal', 'cazorla', 'uveral', 'altamira', 'el calvario', 'el rastro', 'guardatinajas', 'capital urbana calabozo', 'quebrada honda de guache', 'pio tamayo', 'yacambu', 'freitez', 'jose maria blanco', 'concepcion', 'el cuji', 'juan de villegas', 'tamaca', 'aguedo felipe alvarado', 'juarez', 'juan bautista rodriguez', 'cuara', 'diego de lozada', 'paraiso de san jose', 'tintorero', 'jose bernardo dorante', 'coronel mariano peraza ', 'guarico', 'hilario luna y luna', 'humocaro alto', 'humocaro bajo', 'la candelaria', 'cabudare', 'jose gregorio bastidas', 'agua viva', 'sarare', 'buria', 'gustavo vegas leon', 'trinidad samuel', 'camacaro', 'castaneda', 'cecilio zubillaga', 'chiquinquira', 'el blanco', 'espinoza de los monteros', 'lara', 'manuel morillo', 'montana verde', 'montes de oca', 'heriberto arroyo', 'reyes vargas', 'siquisique', 'moroturo', 'xaguas', 'presidente betancourt', 'presidente paez', 'presidente romulo gallegos', 'gabriel picon gonzalez', 'hector amable mora', 'jose nucete sardi', 'pulido mendez', 'la azulita', 'santa cruz de mora', 'mesa bolivar', 'mesa de las palmas', 'canagua', 'capuri', 'chacanta', 'el molino', 'guaimaral', 'mucutuy', 'mucuchachi', 'fernandez pena', 'matriz', 'acequias', 'jaji', 'la mesa', 'san jose del sur', 'tucani', 'florencio ramirez', 'santo domingo', 'las piedras', 'mesa de quintero', 'arapuey', 'palmira', 'san cristobal de torondoy', 'torondoy', 'antonio spinetti dini', 'arias', 'caracciolo parra perez', 'domingo pena', 'el llano', 'gonzalo picon febres', 'jacinto plaza', 'juan rodriguez suarez', 'lasso de la vega', 'mariano picon salas', 'milla', 'osuna rodriguez', 'sagrario', 'los nevados', 'la venta', 'pinango', 'timotes', 'eloy paredes', 'san rafael de alcazar', 'santa elena de arenales', 'santa maria de caparo', 'cacute', 'la toma', 'mucuchies', 'mucuruba', 'geronimo maldonado', 'bailadores', 'tabay', 'chiguara', 'estanquez', 'la trampa', 'pueblo nuevo del sur', 'san juan', 'maria de la concepcion palacios blanco', 'nueva bolivia', 'santa apolonia', 'cano el tigre', 'araguita', 'arevalo gonzalez', 'capaya', 'caucagua', 'panaquire', 'ribas', 'el cafe', 'marizapa', 'cumbo', 'san jose de barlovento', 'el cafetal', 'las minas', 'nuestra senora del rosario', 'higuerote', 'curiepe', 'tacarigua de brion', 'mamporal', 'charallave', 'las brisas', 'altagracia de la montana', 'cecilio acosta', 'los teques', 'el jarillo', 'san pedro', 'tacata', 'paracotos', 'cartanal', 'santa teresa del tuy', 'la democracia', 'ocumare del tuy', 'san antonio de los altos', 'rio chico', 'el guapo', 'tacarigua de la laguna', 'paparo', 'san fernando del guapo', 'santa lucia del tuy', 'cupira', 'machurucuto', 'guarenas', 'san antonio de yare', 'san francisco de yare', 'leoncio martinez', 'petare', 'caucaguita', 'filas de mariche', 'la dolorita', 'cua', 'nueva cua', 'guatire', 'san antonio de maturin', 'san francisco de maturin', 'caripito', 'el guacharo', 'la guanota', 'sabana de piedra', 'san agustin', 'teresen', 'areo', 'capital cedeno', 'san felix de cantalicio', 'viento fresco', 'el tejero', 'punta de mata', 'las alhuacas', 'tabasca', 'temblador', 'alto de los godos', 'boqueron', 'las cocuizas', 'la cruz', 'san simon', 'el corozo', 'el furrial', 'jusepin', 'la pica', 'aparicio', 'aragua de maturin', 'chaguamal', 'el pinto', 'guanaguana', 'la toscana', 'taguaya', 'quiriquire', 'los barrancos de fajardo', 'francisco fajardo', 'guevara', 'matasiete', 'aguirre', 'adrian', 'juan griego', 'yaguaraparo', 'porlamar', 'san francisco de macanao', 'boca de rio', 'los baleales', 'vicente fuentes', 'san juan bautista', 'zabala', 'capital araure', 'rio acarigua', 'capital esteller', 'san jose de la montana', 'san juan de guanaguanare', 'virgen de la coromoto', 'trinidad de la capilla', 'divina pastora', 'pena blanca', 'capital ospino', 'aparicion', 'la estacion', 'payara', 'pimpinela', 'ramon peraza', 'cano delgadito', 'san genaro de boconoito', 'antolin tovar', 'santa fe', 'thermo morles', 'san rafael de palo alzado', 'uvencio antonio velasquez', 'san jose de saguaz', 'villa rosa', 'canelones', 'san isidro labrador', 'san jose de aerocuar', 'tavera acosta', 'rio caribe', 'el morro de puerto santo', 'puerto santo', 'san juan de las galdonas', 'el rincon', 'general francisco antonio vaquez', 'guaraunos', 'tunapuicito', 'santa teresa', 'maracapana', 'el paujil', 'chacopata', 'manicuare', 'tunapuy', 'irapa', 'campo claro', 'maraval', 'san antonio de irapa', 'soro', 'cumanacoa', 'arenas', 'cogollar', 'san lorenzo', 'villa frontado', 'catuaro', 'rendon', 'san cruz', 'santa maria', 'valentin valiente', 'gran mariscal', 'cristobal colon', 
             'bideau', 'punta de piedras', 'guiria', 'rivas berti', 'san pedro del rio', 'palotal', 'general juan vicente gomez', 'isaias medina angarita', 'amenodoro angel lamus', 'la florida', 'boca de grita', 'roman cardenas', 'emilio constantino guerrero', 'monsenor miguel antonio salas', 'la petrolea', 'quinimari', 'bramon', 'cipriano castro', 'manuel felipe rugeles', 'doradas', 'emeterio ochoa', 'san joaquin de navay', 'constitucion', 'la palmita', 'nueva arcadia', 'delicias', 'hernandez', 'la concordia', 'pedro maria morantes', 'dr. francisco romero lobo', 'eleazar lopez contreras', 'juan pablo penalosa', 'potosi', 'araguaney', 'el jaguito', 'la esperanza', 'santa isabel', 'mosquey', 'burbusay', 'general ribas', 'guaramacal', 'vega de guaramacal', 'monsenor jauregui', 'sabana grande', 'cheregue', 'granados', 'arnoldo gabaldon', 'carrillo', 'cegarra', 'chejende', 'manuel salvador ulloa', 'la concepcion', 'cuicas', 'panamericana', 'sabana libre', 'los caprichos', 'el progreso', 'tres de febrero', 'el dividive', 'agua santa', 'agua caliente', 'el cenizo', 'valerita', 'santa maria del horcon', 'el bano', 'jalisco', 'flor de patria', 'la paz', 'pampanito ii', 'betijoque', 'jose gregorio hernandez', 'la pueblita', 'los cedros', 'carvajal', 'campo alegre', 'antonio nicolas briceno', 'jose leonardo suarez', 'sabana de mendoza', 'el paraiso', 'andres linares', 'cristobal mendoza', 'cruz carrillo', 'monsenor carrillo', 'tres esquinas', 'cabimbu', 'jajo', 'la mesa de esnujaque', 'santiago', 'tuname', 'la quebrada', 'juan ignacio montilla', 'la beatriz', 'la puerta', 'mendoza del valle de momboy', 'mercedes diaz', 'caraballeda', 'carayaca', 'caruao chuspa', 'catia la mar', 'el junko', 'la guaira', 'macuto', 'maiquetia', 'naiguata', 'urimare', 'chivacoa', 'temerla', 'san andres', 'yaritagua', 'san javier', 'albarico', 'el guayabo', 'farriar', 'isla de toas', 'monagas', 'san timoteo', 'general urdaneta', 'marcelino briceno', 'manuel guanipa matos', 'ambrosio', 'carmen herrera', 'la rosa', 'german rios linares', 'san benito', 'jorge hernandez', 'punta gorda', 'aristides calvani', 'encontrados', 'udon perez', 'moralito', 'san carlos del zulia', 'santa cruz del zulia', 'urribarri', 'carlos quevedo', 'guamo-gavilanes', 'mariano parra leon', 'jose ramon yepez', 'bari', 'el carmelo', 'potreritos', 'alonso de ojeda', 'venezuela', 'campo lara', 'bartolome de las casas', 'san jose de perija', 'la sierrita', 'las parcelas', 'luis de vicente', 'monsenor marcos sergio godoy', 'tamare', 'antonio borjas romero', 'cacique mara', 'carracciolo parra perez', 'cristo de aranza', 'coquivacoa', 'francisco eugenio bustamante', 'idelfonzo vasquez', 'juana de avila', 'luis hurtado higuera', 'manuel dagnino', 'olegario villalobos', 'venancio pulgar', 'faria', 'ana maria campos', 'donaldo garcia', 'el rosario', 'sixto zambrano', 'el bajo', 'domitila flores', 'francisco ochoa', 'los cortijos', 'marcial hernandez', 'el mene', 'pedro lucas urribarri', 'jose cenobio urribarri', 'rafael maria baralt', 'bobures', 'gibraltar', 'heras', 'monsenor arturo alvarez', 'el batey', 'la victoria', 'raul cuenca', 'sinamaica', 'alta guajira', 'elias sanchez rubio', 'guajira', 'antimano', 'caricuao', 'coche', 'el junquito', 'el valle', 'la vega', 'macarao', 'san bernardino', '23 de enero']

ciudades = ['el cafetal', 'puerto ayacucho', 'atabapo', 'la esmeralda', 'parhueña', 'isla raton', 'barcelona', 'lecheria', 'clarines', 'pariapán', 
            'biruaca', 'maracay', 'colonia tovar', 'ocumare', 'turiamo', 'socopo', 'sabaneta', 'barinitas', 'guayana', 'puerto ordaz', 'ciudad bolivar',
            'upata', 'tumeremo', 'el callao', 'guacipati', ' uairen', 'caicara', 'maripa', 'las claritas', ' moron ', 'bejumá', 'sierra imataca', 
            'piacoa', 'sacupana', 'capure', 'araguaimujo', 'macareo', 'mirimire', 'adicora', 'valle la pascua', 'cubiro', ' osma', ' chuspa', 
            'carmen de uria', 'el vigia', ' tabay', 'higuerote', 'caucagua', 'capayacuar', 'macanao', 'acarigua', 'cariaco', 'yaguaraparo', 'tunapuy', 
            ' irapa', ' araya', 'capacho', 'bocono', ' monay', 'ciudad ojeda', 'machiques']

venezuela = estados + municipios + parroquias + ciudades + ['venezuela', 'venezolan', 'vzla']
vzla = r'|'.join(venezuela)

signos = r'[@#\-_"!\¡\?\(\)\/\|*]'

redes = ['facebook', 'instagram', 'x.com', 'tiktok', 'threads', 'youtube']

<h3>Limpieza y normalización</h3>

In [4]:
dr2 = dr.copy()
dr2[['title', 'snippet_full']] = dr2[['title', 'snippet_full']].astype(str).map(lambda x: x.strip()) # Asegurar string y limpiar espacios en blanco
dr2 = dr2[~dr2['url'].str.contains(r'jabertur.net')] # Eliminar ciertas url comprobadas
dr2 = dr2[(dr2['snippet_full'] != '') & (dr2['snippet_full'] != 'nan') &(dr2['snippet_full'] != None) & (dr2['snippet_full'] != 'None')]
dr2[['title_clean', 'snippet_full_clean']] = dr2[['title', 'snippet_full']].map(lambda x: unidecode(x.lower())) # Eliminar acentos, signos raros, emojis
dr2['snippet_full_clean'] = dr2['snippet_full_clean'].str.replace(signos, ' ', regex=True) # Eliminar ciertos signos de puntuación
dr2['snippet_full_clean'] = dr2['snippet_full_clean'].str.replace(r'[.,]', '', regex=True) # Eliminar puntos y comas atendiendo a números
dr2['snippet_full_clean'] = dr2['snippet_full_clean'].str.replace(r'\s+', ' ', regex=True).str.strip() # Eliminar espacios dobles

<h3>Filtrado de registros que no corresponden a venezuela</h3>

In [5]:
dr2 = dr2[dr2['snippet_full_clean'].str.contains(vzla, case=False, na=False) & ~dr2['snippet_full_clean'].str.contains(amer, case=False, na=False)]
print(dr2.shape)

(46, 22)


In [ ]:
# dr2.sample(10)

<h3>Clasificación de registros por red social, tipo de ambiente, flujo y medio de transporte</h3>

In [6]:
import pandas as pd
import re

print('Número de registros:', dr2.shape, '\n')

# Frecuencia por red (Se mantiene igual, útil para monitoreo de emergencias)
red = f"({'|'.join(redes)})"
dr2['red_social'] = dr2['url'].str.extract(red, expand=False, flags=re.IGNORECASE)
dr2['red_social'] = dr2['red_social'].fillna('web').str.lower()
regxred = dr2['red_social'].value_counts()

print('Registros por red social')
print(regxred.to_string(header=False), '\n'*2)

# Clasificación por tipo de zona o impacto (Adaptado del clasificador original)
def clasificar_ambiente(texto, lista_palabras):
    if pd.isna(texto):
        return ''
    pat = []
    for p in lista_palabras:
        if p in texto:
            pat.append(p)
    if pat:
        return pat
    return ''

# Se asume que ahora existe un diccionario 'emergencias' (antes 'turismo')
# con las nuevas palabras clave para cada categoría.
for zona, palabras in emergencias.items():
    dr2[zona] = dr2['snippet_full_clean'].apply(lambda x: clasificar_ambiente(x,  palabras))

# Resumen por zona, nivel de afectación y medios de rescate/transporte
# Se reemplazaron los ambientes turísticos por zonas geográficas y vulnerables
zona = ['capital', 'andes', 'oriente', 'occidente', 'llanos', 'costa', 'barrio', 'urbanizacion']
# Se reemplazó hospedaje/agencia/gasto por el impacto del evento
afectacion = ['daños', 'victimas', 'rescate']
# Se mantienen los medios, pero enfocados en logística de emergencia y rescate
medio = ['aereo', 'acuatico', 'terrestre']

atencion = emergencias['atencion'] # Antes aloja (hospedaje) -> ahora refugios/hospitales
eventos = emergencias['eventos']   # Antes actividad -> ahora sismos/lluvias/deslaves

conteo = dr2[zona].apply(lambda x: x != '').sum().sort_values(ascending=False).rename(str.capitalize)
conteo2 = dr2[afectacion].apply(lambda x: x != '').sum().sort_values(ascending=False).rename(str.capitalize)
conteo3 = dr2[medio].apply(lambda x: x != '').sum().sort_values(ascending=False).rename(str.capitalize)

def frecuencias(df, columna, cambiar=False):
    explot = df[columna].explode()
    limpio = explot.dropna().astype(str).str.strip().str.lower()
    
    if cambiar:
        # emergencias médicas, refugios y desastres
        if columna == 'atencion':
            equiv = {'refugio':['refugio', 'albergue', 'carpa', 'campamento'], 
                     'hospital':['hospital', 'clinica', 'cdi', 'ambulatorio', 'barrio adentro', 'triaje'], 
                     'centro_acopio':['acopio', 'donaciones', 'recoleccion']}
        elif columna == 'eventos':
            equiv = {'sismo':['tembl', 'tiembla', 'terremoto', 'replica', 'sismo'], 
                     'inundacion':['inund', 'aneg', 'desbord', 'crecida'], 
                     'deslave':['deslave', 'derrumbe', 'deslizamiento', 'alud', 'barro'], 
                     'rescate':['rescate', 'evacu', 'salvamento', 'busqueda'], 
                     'colapso':['colapso', 'caida', 'rotura', 'falla', 'sin servicio'],
                     'lluvias':['aguacero', 'tormenta', 'precipitaciones', 'vaguada']}
        # logística de rescate y protección civil
        elif columna == 'aereo':
            equiv = {'helicoptero':['helicoptero', 'super puma', 'cougar'], 
                     'avion':['avion', 'hercules', 'carguero'], 
                     'dron':['dron', 'drone', 'uav']}
        elif columna == 'acuatico':
            equiv = {'rescate':['lancha', 'zodiac', 'bote', 'salvavidas'], 
                     'masivo':['buque', 'patrullera', 'fragata'], 
                     'rural':['peñero', 'curiara', 'canoa']}
        elif columna == 'terrestre':
            equiv = {'emergencia':['ambulancia', 'paramedicos', 'bomberos'], 
                     'maquinaria':['retroexcavadora', 'tractor', 'payloader', 'grua'], 
                     'rescate':['rustico', 'jeep', 'machito', '4x4', 'proteccion civil'], 
                     'masivo':['autobus', 'convoy', 'camion', 'carguero']}
        
        for k, v in equiv.items():
            mask = limpio.isin(v)
            limpio.loc[mask] = k    
            
    conteo = limpio[limpio != ""].value_counts()
    conteo.index = conteo.index.str.capitalize()
    return conteo

conteo4 = frecuencias(dr2, 'aereo')
conteo5 = frecuencias(dr2, 'acuatico') # Antes 'maritimo'
conteo6 = frecuencias(dr2, 'terrestre')

precont = pd.concat([conteo4, conteo5, conteo6]).groupby(level=0).sum()

# Ajuste de normalización de términos para el reporte top 10
cambios = {'Jeep': 'Rustico', 'Machito': 'Rustico', 'Zodiac': 'Lancha', 
           'Payloader': 'Retroexcavadora', 'Super puma': 'Helicoptero', 'Convoy': 'Camion'}
eliminar = ['Terminal', 'Estacion', 'Taller']

conteo7 = precont.rename(index=cambios).groupby(level=0).sum()
conteo7 = conteo7.drop(labels=eliminar, errors='ignore')
conteo7 = conteo7.sort_values(ascending=False)[:10]

# Llamadas con las variables renombradas (atencion y eventos)
conteo8 = frecuencias(dr2, 'atencion', True)
conteo9 = frecuencias(dr2, 'eventos', True)

print('Zonas Afectadas y Entornos')
print(conteo, '\n'*2)
print('Nivel de Afectación (Daños, Víctimas, Rescates)')
print(conteo2, '\n'*2)
print('Categorías de Logística y Transporte Empleado')
print(conteo3, '\n'*2)
print("Vehículos y Maquinaria (Top 10):")
print(conteo7.to_string(header=False), '\n'*2)
print('Logística Aérea')
print(conteo4.to_string(header=False), '\n'*2)
print('Logística Acuática')
print(conteo5.to_string(header=False), '\n'*2)
print('Logística Terrestre / Maquinaria')
print(conteo6.to_string(header=False), '\n'*2)
print('Centros de Atención y Refugios')
print(conteo8.to_string(header=False), '\n'*2)
print('Tipos de Eventos y Emergencias')
print(conteo9.to_string(header=False))

Número de registros: (46, 22) 

Registros por red social
web         45
facebook     1 


Zonas Afectadas y Entornos
Oriente         17
Llanos          13
Capital          7
Costa            7
Andes            5
Occidente        2
Barrio           0
Urbanizacion     0
dtype: int64 


Nivel de Afectación (Daños, Víctimas, Rescates)
Victimas    3
Rescate     2
Daños       1
dtype: int64 


Categorías de Logística y Transporte Empleado
Aereo        0
Acuatico     0
Terrestre    0
dtype: int64 


Vehículos y Maquinaria (Top 10):
Series([], ) 


Logística Aérea
Series([], ) 


Logística Acuática
Series([], ) 


Logística Terrestre / Maquinaria
Series([], ) 


Centros de Atención y Refugios
Hospital         1
Alimentos        1
Iglesia          1
Centro_acopio    1 


Tipos de Eventos y Emergencias
Inundacion       8
Lluvia           8
Sismo            8
Deslave          3
Onda tropical    1
Anegacion        1
Lluvias          1


In [ ]:
# dr2[['title', 'url', 'snippet_full', 'montaña', 'paramo', 'selva', 'llano', 'playa', 'lago', 'rio', 'desierto', 'ciudad', 'hospedaje', 'agencia', 'gasto']].sample(100)

<h3>NER - Reconocimiento de Entidades</h3>

In [7]:
from collections import Counter
import spacy
import re

# Modelo grande en español: CMD -> spacy download es_core_news_lg
print('Cargando el modelo...')
nlp = spacy.load('es_core_news_lg')

def extraer_entidades(columna):
    lugares_por_celda = []
    actores_por_celda = [] # Nueva lista para autoridades, instituciones y personas
    
    # Procesar por lotes es más eficiente con nlp.pipe
    for doc in nlp.pipe(columna.astype(str), batch_size=50):
        lugares = []
        actores = []
        
        for ent in doc.ents:
            # Extraer lugares (Geografía, ciudades, estados)
            if ent.label_ == 'LOC':
                lugares.append(ent.text)
            # Extraer actores (PER = Personas/Autoridades, ORG = Organizaciones/Instituciones/ONGs)
            elif ent.label_ in ['PER', 'ORG']:
                actores.append(ent.text)
                
        lugares_por_celda.append(lugares if lugares else "")
        actores_por_celda.append(actores if actores else "")
        
    return lugares_por_celda, actores_por_celda

print('Iniciando el reconocimiento de entidades (Lugares y Actores/Autoridades)...')

# Asignamos los resultados a dos columnas distintas simultáneamente
dr2['lugares'], dr2['autoridades_actores'] = extraer_entidades(dr2['snippet_full_clean'])

print('Terminado.')

Cargando el modelo...
Iniciando el reconocimiento de entidades (Lugares y Actores/Autoridades)...
Terminado.


<h3>Limpieza final y resumen de lugares por frecuencia (NER)</h3>

In [8]:
# Palabras a excluir de los resultados
blacklist = ['nan', 'venezuela', 'vzla', 'posts', 'ubicacion', 'siguenos', 'descubre', 'encuentra', 'visitanos', 'hospedaje', 'reserva', 'reservas',
             'tripadvisor', 'kayak', 'urb', 'edo', '☎', '🈴', 'yeiber', 'av', 'aqui', 'airbnb', 'urb', 'book homes', 'ensueno', 'rios', 'town house',
             'ninas', 'jeep', 'cabana', 'rio smart', 'camaras', 'alli', 'armonia', 'zelle', 'audubon de', 'audubon de venezuela', 'closet', 'travesia',
             'hotel', 'republica bolivariana de', 'republica bolivariana de venezuela', 'asiaafrica', 'en estados unidos', 'munecas', 'golfo de', 
             '!unete', 'la av', 'gavidia', 'jhonathan miranda', 'paseo', 'avion', 'lian', 'tealca', 'Meridalosandesverdegreen', 'metro chorrillos', 
             'lavanderia', 'copa venezuela bahia', 'de colombia', 'playa de', 'edo.', 'pais', 'sabado', '1 vzla', 'youtube', 'ovr7', 'miss', 'ninos',
             'caracas maracay', 'mister', 'tours', '6:30 am', 'alla', 'montanabeach', 'meridalosandesverdegreen', 'parrillera', 'merida c.a', 'carmen',
             'bano', 'unete', 'mar!', 'primaderm', 'senora', 'venaventours', 'urb.', 'soygruporoelroel', 'millas', '!reserva', 'maracay valencia', 
             'travel', 'turismovenezuela', 'luchaalmada15 leticiagomezve', 'conocevenezuela', 'johancolmenaresc', 'venezolano', 'carabobo aragua', 
             'caracas aragua', '0424 3109693 barquisimeto', 'kayakvenezuela', 'leticiagomezve', 'on july', 'minturvzla', 'av r 8', 'Lian',
             'parapentemeridacom', 'turis', 'caracas valencia', 'adventures', 'santorinibdu', 'enamorate siguenos', 'paseo', 'tepuy travel', 
             'minimalfurgo', 'meridaven', 'tuscany', 'via guarico sector', 'cagua maracay valencia', 'venezuela unete', 'town house urbanizacion', 
             'aragua carabobo', 'santiago', 'relax', 'merida caracas', 'aguila', 'aguilas', 'cascadadelvino', 'caracas valencia travel', 'roque', 
             'maracay caracas', 'laromerena on instagram', 'expo', 'venezuela motoadventurecamp', 'missvenezuela', 'lena', 'guadalupe',
             'santa rita montereal hotel', 'cienagamia', 'fullday', 'mara', 'sierra de santiago nl']

# Palabras o expresiones a cambiar por un equivalente
equivalencias = {'parque n morrocoy':'morrocoy', 'parque nacional morrocoy':'morrocoy', 'parque nacional canaima': 'canaima', 'mcbo': 'maracaibo', 
                 'isla de margarita': 'margarita', 'gran caracas': 'caracas', 'andes': 'los andes', 'manantial valencia': 'valencia', 
                 'ordaz': 'pto. Ordaz', 'parque nacional el avila': 'el avila', 'avila': 'el avila', 
                 'parque nacional mochima': 'mochima', 'parque nacional sierra de la culata': 'sierra Nevada', 'ee.uu': 'eeuu', 'ee.uu.':'eeuu', 
                 'parque nacional sierra nevada': 'sierra nevada', 'caracas-vzla': 'caracas', 'isla margarita': 'margarita', 
                 'puerto cruz': 'pto. la cruz', 'estados unidos': 'eeuu', 
                 'montanas': 'montaña', 'merida!!': 'merida', 'cienaga': 'la cienaga', 'bahia de pampatar #escamas #bodegon': 'bahia de pampatar', 
                 'merida and more': 'merida', 'losroques': 'los roques', 
                 'archipielago de los roques': 'los roques', 'Aragua turismoenaraguave on instagram': 'aragua', 'venezuela caracas': 'caracas', 
                 'valencia carabobo': 'valencia', 'san cristobal tachira': 'tachira', 'realtre caracas': 'caracas', 'venezuela merida': 'merida', 
                 'cienaga aragua': 'la cienaga', 'carabobo valencia': 'valencia', 'aragua maracay': 'maracay', 
                 'san juan de los morro': 'san juan de los morros', 'playasdevenezuela falcon': 'falcon', 
                 'anzoategui venezuela descripcion': 'anzoategui', 'team caracas': 'caracas', 'nirgua edo yaracuy': 'nirgua', 'merida merida': 'merida',
                 'maracay aragua': 'maracay', 'merida venezuela on instagram': 'merida', 'hotelbroadway': 'hotel broadway', 
                 'chichiriviche falcon': 'chichiriviche', 'lacienaga': 'la cienaga', 'chichiriviche edo falcon': 'chichiriviche', 
                 'archipielago de losroques': 'los roques', 'de valencia': 'valencia', 'aragua turismoenaraguave on instagram': 'aragua', 
                 'coro falcon': 'coro', 'parque nacional canaima': 'canaima', 'parque nacional medanos de coro': 'medanos de coro', 
                 'san felipe yaracuy': 'san felipe', 'pico avila': 'el avila', 'bahia de pampatar escamas bodegon restaurant': 'bahia de pampatar',
                 'ojeda': 'ciudad ojeda', 'tinaquillo edo cojedes': 'tinaquillo', 'atvutvvalera atv barinas': 'barinas', 'orinoco':'rio orinoco'}  
                 # '':'', '':'', '':'', '':'', }

def limpiar_entidades(lista_lugares):
    if not isinstance(lista_lugares, list):
        return []
    
    limpios = []
    for lugar in lista_lugares:
        lugar = unidecode(lugar.lower().strip())
        lugar = equivalencias[lugar] if lugar in equivalencias else lugar
        lugar = lugar.replace('estado ', '') if lugar.startswith('estado ') else lugar
        lugar = lugar.replace('edo ', '') if lugar.startswith('edo ') else lugar
        lugar = lugar.replace(' venezuela', '') if lugar.endswith(' venezuela') else lugar
        lugar = lugar.replace(' vzla', '') if lugar.endswith(' vzla') else lugar
        lugar = lugar.replace('#', '') if lugar.startswith('#') else lugar
        lugar = lugar.replace(' #', '') if lugar.endswith(' #') else lugar

        if (lugar not in blacklist and       # Que no esté en la blacklist
            len(lugar) > 3 and not           # Que tenga más de 3 caracteres (evita emojis, siglas, abreviaturas)
            re.match(r'^[^\w\s]+$', lugar)):  # Que no sea solo números o símbolos

            lugar = lugar.strip().capitalize()
            limpios.append(lugar)

    return limpios

dr2['lugares_clean'] = dr2['lugares'].apply(limpiar_entidades)
todos_los_lugares = [lug for sublista in dr2['lugares_clean'] for lug in sublista]
frecuencias = Counter(todos_los_lugares).most_common()

In [9]:
# Resultados por estado
print(f'Total de lugares detectados: {len(todos_los_lugares)}', '\n')
set_estados = set(estados)
print('Estados por frecuencia (Top 10):')
for lugar, count in frecuencias[:16]:
    if lugar.lower() == 'caracas':
        print(f"Distrito capital: {count}")
    if lugar.lower() in set_estados:
        print(f"{lugar}: {count}")

todos_los_actores = []
for c in dr2['autoridades_actores']:
    if isinstance(c, list):  # Verificamos que sea una lista y no un string vacío
        todos_los_actores.extend(c)

# 2. Contar la frecuencia de cada actor/institución
frecuencias_actores = Counter(todos_los_actores).most_common()

print(f'Total de menciones a autoridades/actores detectados: {len(todos_los_actores)}', '\n')
print('Autoridades, Instituciones y Actores (Top 15):')

# 3. Mostrar los resultados (puedes cambiar el 15 por el número que desees)
for actor, count in frecuencias_actores[:15]:
    # .title() ayuda a que los nombres se vean presentables (ej: "proteccion civil" -> "Proteccion Civil")
    print(f"{actor.title()}: {count}")

print('\n' * 2)

Total de lugares detectados: 64 

Estados por frecuencia (Top 10):
Nueva esparta: 15
Barinas: 8
Distrito capital: 5
La guaira: 3
Sucre: 2
Tachira: 2
Amazonas: 1
Bolivar: 1
Delta amacuro: 1
Total de menciones a autoridades/actores detectados: 20 

Autoridades, Instituciones y Actores (Top 15):
Hara: 2
San Geronimo De Guayabal: 1
Henry Baloa: 1
Canciller Yvan Gil: 1
25 Jun: 1
Delcy Rodriguez: 1
Ate Ao: 1
Lusodescendentes: 1
Aldo Pusticcio: 1
Reinaldo Silva Reinaldosilva119 Gmailcom: 1
Reinaldo Silva5: 1
Zodi Nueva Esparta: 1
Ana Carolina Arias Nueva Esparta: 1
Meteorological: 1
Bolivar: 1





In [10]:
# Resultados por destino
print('Destinos por frecuencia (Top 100):')
for lugar, count in frecuencias[:100]:
    if lugar.lower() not in set_estados: 
        print(f"{lugar}: {count}")

Destinos por frecuencia (Top 100):
Caracas: 5
Mar la maxima: 2
Rio orinoco: 2
Anzoategui carabobo monagas: 1
Santa cruz de la sierra: 1
2 sept: 1
Carolina jaimes branger: 1
Republica nicolas maduro: 1
San juan municipio diaz: 1
Island nueva esparta: 1
9 jun: 1
Universal nueva esparta: 1
Margarita: 1
Porlamar juan: 1
Isla de coche: 1
Bolivar segun: 1
Garcia: 1
Pabellon: 1
Bolivar amazonas llanos centrales apure: 1
Lara falcon los andes: 1
1 may: 1
Santa rosa municipio rojas: 1


Registros de Venezuela

In [11]:
dr3 = dr2[dr2['snippet_full'].str.contains(vzla) & ~dr2['snippet_full'].str.contains(amer)]
dr3.shape

(7, 41)

In [ ]:
# dr3.to_excel('a_revisar_turismo_solo_vzla.xlsx', index=False)

Registros de Venezuela que solo contienen palabras de turismo: para analizar paquetes

In [ ]:
dr4 = dr2[dr2['snippet_full'].str.contains(vzla) & ~dr2['snippet_full'].str.contains(amer) & dr2['snippet_full'].str.contains(paq)]
dr4.shape

In [ ]:
# dr4.to_excel('a_revisar_turismo_vzla_previo.xlsx', index=False)

In [ ]:
dr2.head(2)

Búsqueda de otros países

In [12]:
dr5 = dr2[~dr2['snippet_full'].str.contains(vzla)]
dr5.shape

(39, 41)

In [ ]:
# dr5.to_excel('a_revisar_turismo_otros_paises.xlsx', index=False)

Lugares en blanco

In [13]:
dr6 = dr2[~dr2['lugares'].astype(bool)]
dr6.shape

(8, 41)

In [ ]:
# dr6.to_excel('a_revisar_turismo_lugares_blanco.xlsx', index=False)

<h3>Guardar df</h3>

In [41]:
# dr2.to_excel('vzla_turismo_scrap_categorizada.xlsx', index=False)